# ROGII Wellbore Geology - LSTM Training

Train a Bidirectional LSTM to predict **TVT** from horizontal well logs.

**Features (6):** `MD, X, Y, Z, GR, TVT_input` (Masked for missing values)
**Label:** `TVT`

**Runtime**: Kaggle GPU (T4/P100) - Keras 3 + JAX backend

**Author**: Samir Attrah

In [2]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
jax.config.update("jax_enable_x64", True)

import keras
from keras import layers, callbacks, regularizers
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings, random
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Keras backend : {keras.backend.backend()}")
print(f"Working dir   : {os.getcwd()}")


Keras version : 3.12.0
Keras backend : jax
Working dir   : /home/samer/Documents/competitions/ROGII/notebooks


In [3]:
# Cell 2: Auto-detect dataset path
def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]
    for path in candidates:
        if os.path.isdir(path):
            if "train" in os.listdir(path): return path
    raise FileNotFoundError("Dataset not found.")

DATA_DIR = find_data_dir()


In [4]:
# Cell 3: Configuration
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else "/home/samer/Documents/competitions/ROGII/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "seed": 42,
    "data_dir": DATA_DIR,
    "model_path": f"{OUT_DIR}/lstm_tvt_model.keras",
    "scaler_path": f"{OUT_DIR}/scaler_params.pkl",
    "window_size": 2,
    "stride": 1,
    "mask_value": -10.0,
    "lstm_units_1": 32,
    "lstm_units_2": 32,
    "lstm_units_3": 32,
    "lstm_units_4": 16,
    "lstm_units_5": 16,
    "lstm_units_6": 16,
    "lstm_units_7": 8,
    "lstm_units_8": 8,
    "kr_rate": 7e-5,
    "dense_units": 6,
    "dropout": 0.40,
    "epochs": 50,
    "batch_size": 8,
    "lr": 1e-3,
    "val_ratio": 0.20,
    "max_wells": None,
    "gcn": 1,
}

FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TARGET = "TVT"
MASK_VALUE = CONFIG["mask_value"]


In [5]:
# Cell 4: Data helpers
def well_ids(data_dir, split="train"):
    pattern = os.path.join(data_dir, split, "*__horizontal_well.csv")
    return sorted(os.path.basename(f).split("__")[0] for f in glob.glob(pattern))

def load_well(data_dir, wid, split="train"):
    path = os.path.join(data_dir, split, f"{wid}__horizontal_well.csv")
    return pl.read_csv(path, infer_schema_length=10000)

def preprocess(df):
    for col in FEATURE_COLS:
        if col in df.columns:
            df = df.with_columns(pl.col(col).interpolate())
    return df

def make_seqs(feats, tgts, ws, stride):
    idx = range(0, len(feats) - ws + 1, stride)
    X = np.empty((len(idx), ws, feats.shape[1]), dtype=np.float64)
    y = np.empty(len(idx), dtype=np.float64)
    for i, s in enumerate(idx):
        X[i] = feats[s:s+ws]
        y[i] = tgts[s+ws-1]
    return X, y


In [6]:
# Cell 5: Prepare training data (Masked float64 SCALE)
def prepare_data(cfg):
    ids_all = well_ids(cfg["data_dir"], "train")
    if cfg["max_wells"]: ids_all = ids_all[:cfg["max_wells"]]
    ws = cfg["window_size"]

    np.random.seed(cfg.get("seed", 42))
    n_val = max(1, int(len(ids_all) * cfg["val_ratio"]))
    val_set = set(np.random.permutation(len(ids_all))[:n_val])
    
    Xt_raw, yt_raw, Xv_raw, yv_raw = [], [], [], []
    for i, wid in enumerate(ids_all):
        try:
            df = preprocess(load_well(cfg["data_dir"], wid))
            df = df.filter(pl.col(TARGET).is_not_null())
            if len(df) <= ws: continue
            f = df.select(FEATURE_COLS).to_numpy().astype(np.float64)
            t = df.select(TARGET).to_numpy().ravel().astype(np.float64)
            X, y = make_seqs(f, t, ws, cfg["stride"])
            (Xv_raw if i in val_set else Xt_raw).append(X)
            (yv_raw if i in val_set else yt_raw).append(y)
        except:
            pass

    Xt_all = np.concatenate(Xt_raw)
    yt_all = np.concatenate(yt_raw)
    Xv_all = np.concatenate(Xv_raw)
    yv_all = np.concatenate(yv_raw)

    f_flat = Xt_all.reshape(-1, len(FEATURE_COLS))
    feat_mean = jnp.nanmean(jnp.array(f_flat, dtype=jnp.float64), axis=0)
    feat_std = jnp.nanstd(jnp.array(f_flat, dtype=jnp.float64), axis=0)
    feat_mean = jnp.where(jnp.isnan(feat_mean), 0.0, feat_mean)
    feat_std = jnp.where(jnp.isnan(feat_std) | (feat_std == 0), 1.0, feat_std)
    
    target_mean = jnp.nanmean(jnp.array(yt_all, dtype=jnp.float64))
    target_std = jnp.nanstd(jnp.array(yt_all, dtype=jnp.float64))
    target_mean = jnp.where(jnp.isnan(target_mean), 0.0, target_mean)
    target_std = jnp.where(jnp.isnan(target_std) | (target_std == 0), 1.0, target_std)

    # Normalize and then mask
    X_train_n = (jnp.array(Xt_all, dtype=jnp.float64) - feat_mean) / feat_std
    y_train_n = (jnp.array(yt_all, dtype=jnp.float64) - target_mean) / target_std
    X_val_n = (jnp.array(Xv_all, dtype=jnp.float64) - feat_mean) / feat_std
    y_val_n = (jnp.array(yv_all, dtype=jnp.float64) - target_mean) / target_std

    X_train_n = jnp.where(jnp.isnan(X_train_n), MASK_VALUE, X_train_n)
    X_val_n = jnp.where(jnp.isnan(X_val_n), MASK_VALUE, X_val_n)
    y_train_n = jnp.where(jnp.isnan(y_train_n), 0.0, y_train_n)
    y_val_n = jnp.where(jnp.isnan(y_val_n), 0.0, y_val_n)

    scaler = {"feature_cols": FEATURE_COLS, "feat_mean": np.array(feat_mean), "feat_std": np.array(feat_std), "target_mean": float(target_mean), "target_std": float(target_std), "mask_value": MASK_VALUE, "normalized": True}
    return np.array(X_train_n, dtype=np.float32), np.array(y_train_n, dtype=np.float32), np.array(X_val_n, dtype=np.float32), np.array(y_val_n, dtype=np.float32), yv_all, scaler

print("--- Loading and preparing training data (Masked float64) ---")
X_train, y_train, X_val, y_val, y_val_raw, scaler = prepare_data(CONFIG)
with open(CONFIG["scaler_path"], "wb") as f:
    pickle.dump(scaler, f)


--- Loading and preparing training data (Masked float64) ---


In [6]:
# Cell 6: Build & train BiLSTM model
def build_model(input_shape, cfg):
    inp = keras.Input(shape=input_shape)
    x = layers.Masking(mask_value=cfg["mask_value"])(inp)
    x = layers.LSTM(cfg["lstm_units_1"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_2"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_3"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_4"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_5"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_6"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_7"], return_sequences=True, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.LSTM(cfg["lstm_units_8"], return_sequences=False, kernel_regularizer=regularizers.L2(cfg["kr_rate"]),dropout=cfg["dropout"])(x)
    x = layers.Dense(cfg["dense_units"], activation="relu")(x)
    x = layers.Dropout(cfg["dropout"])(x)
    out = layers.Dense(1, activation="linear")(x)
    m = keras.Model(inp, out, name="BiLSTM_TVT")
    m.compile(optimizer=keras.optimizers.Adam(cfg["lr"], global_clipnorm=cfg["gcn"]), loss="mse", metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
    return m

model = build_model((CONFIG["window_size"], len(FEATURE_COLS)), CONFIG)
cbs = [callbacks.ModelCheckpoint(CONFIG["model_path"], monitor="val_rmse", save_best_only=True, mode="min"), callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=3, min_lr=1e-6)]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=CONFIG["epochs"], batch_size=CONFIG["batch_size"], callbacks=cbs)


In [7]:
# Cell 7: Evaluate
best_model = keras.saving.load_model(CONFIG["model_path"])
target_mean = jnp.array(scaler["target_mean"], dtype=jnp.float64)
target_std = jnp.array(scaler["target_std"], dtype=jnp.float64)
yp_n = best_model.predict(X_val, batch_size=512).ravel()
yp = (jnp.array(yp_n, dtype=jnp.float64) * target_std) + target_mean
rmse = float(np.sqrt(np.mean((np.array(yp) - y_val_raw) ** 2)))
print(f"\nVal RMSE (RAW SCALE): {rmse}")


In [ ]:
# Cell 8: Plot Training History
import matplotlib.pyplot as plt
import os

ANALYTICS_DIR = "../analytics"
os.makedirs(ANALYTICS_DIR, exist_ok=True)

def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    ax1.plot(history.history['loss'], label='Train')
    ax1.plot(history.history['val_loss'], label='Val')
    ax1.set_title('Model Loss (MSE)')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(True)
    
    # RMSE
    ax2.plot(history.history['rmse'], label='Train')
    ax2.plot(history.history['val_rmse'], label='Val')
    ax2.set_title('Model RMSE')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('RMSE')
    ax2.legend(); ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYTICS_DIR, "training_history.png"))
    plt.show()

if 'history' in locals():
    plot_history(history)


In [ ]:
# Cell 9: Predicted vs Actual (Validation Set)
def plot_pred_vs_actual(y_true, y_pred):
    plt.figure(figsize=(10, 6))
    plt.scatter(y_true, y_pred, alpha=0.1, s=1, color='blue')
    
    # Perfect prediction line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    
    plt.title(f'Actual vs Predicted TVT (Val Set)\nRMSE: {rmse:.2f}')
    plt.xlabel('Actual TVT'); plt.ylabel('Predicted TVT')
    plt.legend(); plt.grid(True)
    
    plt.savefig(os.path.join(ANALYTICS_DIR, "val_pred_vs_actual.png"))
    plt.show()

if 'y_val_raw' in locals() and 'yp' in locals():
    plot_pred_vs_actual(np.array(y_val_raw), yp)
